# KI in der Medizin: Chancen und Limitierungen


Wir vergleichen acht Gewebeklassen mit MobileNetV2. Labels in diesem Lehrdatensatz sind keine klinischen Diagnosen. Wir verändern Datenverteilung und Fehlergewichtung und untersuchen, was sich an den Vorhersagen ändert.

Die technischen Funktionen sind bereitgestellt. Drei kurze Codeaufgaben übertragen bekannte Ideen aus dem Grundlagenkurs. Hinweise und Lösungen lassen sich ausklappen.

## 1. Start und Arbeitsweise

Notebook von oben nach unten bearbeiten, Modellläufe nur nach Absprache starten. Vor jedem Versuch eine Vermutung notieren und nachher mit den Ergebnissen vergleichen. Im Zweierteam Bedienung und Erklärung abwechseln.

**Training über Nacht:** VS Code und Kernel offen lassen, Laptop am Netzteil und mit offenem Deckel stehen lassen, nicht herunterfahren oder neu starten. Windows+L zum Sperren ist erlaubt. Der Energieschutz wirkt nur während eines gestarteten Trainings. Eine laufende Zelle blockiert weitere Codezellen in demselben Kernel. Eine einzelne Zelle startet keine späteren Versuche automatisch. „Alle ausführen“ kann die vier Läufe nacheinander starten, aber nur nach Freigabe durch die Leitung.

**Gespeicherter Stand:** Nach jeder abgeschlossenen Epoche wird das Modell mit Verlauf gespeichert. Nach Neustart dieselben Einstellungen und denselben Laufnamen verwenden. Die laufende Epoche kann bei einem Abbruch verloren gehen. Ergebnisse werden nie als vorhanden ausgegeben, wenn der passende Modelllauf fehlt.

In [ ]:
from pathlib import Path
import sys
for basis in (Path.cwd(), *Path.cwd().parents):
    kandidaten = (basis, basis / "science-college-ki-medizin-students")
    projekt = next((p for p in kandidaten if (p / "src" / "hauptversuch.py").is_file()), None)
    if projekt is not None:
        break
else:
    raise FileNotFoundError("Den vollständigen Schüler-Projektordner öffnen.")
for ordner in (projekt / "src", projekt / "notebooks"):
    sys.path.insert(0, str(ordner))
import numpy as np
import tensorflow as tf
import hauptversuch as hv
from versuchsbetrieb import lade_histologie, modelllauf, werte_aus, kontrolliere

# Gemeinsame Einstellungen. Änderungen erfordern neue Laufnamen.
EPOCHEN = 30
TRAININGSANTEIL = 0.75
FREIE_SCHICHTEN = 3
LAUFGRUPPE = "workshop_v1"
# True erst nach Absprache. False lädt ausschließlich abgeschlossene eigene Läufe.
TRAINING_STARTEN = False

def lauf(name, x, y, **kwargs):
    return modelllauf(projekt, LAUFGRUPPE + "_" + name, x, y,
                      daten["x_val"], daten["y_val"], epochs=EPOCHEN,
                      num_unfrozen_layers=FREIE_SCHICHTEN,
                      training_erlaubt=TRAINING_STARTEN, **kwargs)


## 2. Gewebebilder und drei getrennte Datensätze

Wir nutzen den lokal vorbereiteten Datensatz `colorectal_histology`. Seine acht Labels unterscheiden Gewebetypen und Hintergründe. Betrachtet die unten ausgegebenen Namen und Beispiele: Welche Unterschiede erkennt ihr?

**Drei Rollen:** Training passt Gewichte an, Validierung begleitet das Training, Testbilder werden erst zur Auswertung verwendet. Nach wiederholten Testvergleichen ist der Test eine interne Workshop-Bewertung und keine neue unabhängige klinische Prüfung.

**One-Hot-Codierung:** Bei acht Klassen ist Label 2 der Vektor `[0, 0, 1, 0, 0, 0, 0, 0]`. Die 1 steht an Position 2, weil Python bei 0 beginnt. Die Umwandlung übernimmt der bereitgestellte Trainingscode. Auch Skalierung, Bildvergrößerung und Datenerweiterung sind vorbereitet.

In [ ]:
bilder, labels, klassennamen = lade_histologie(projekt)
daten = hv.split_data(bilder, labels, subset_frac=TRAININGSANTEIL)
print("Klassen:", list(enumerate(klassennamen)))
for rolle in ("train", "val", "test"):
    print(rolle, len(daten["y_" + rolle]),
          np.bincount(daten["y_" + rolle], minlength=8))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for klasse, ax in enumerate(axes.flat):
    index = np.flatnonzero(labels == klasse)[0]
    ax.imshow(bilder[index]); ax.set_title(klassennamen[klasse]); ax.axis("off")
plt.tight_layout(); plt.show()


## 3. Baseline: Modell starten und Ergebnisse lesen

Ein vortrainiertes MobileNetV2 liefert Bildmerkmale. Darauf folgen eine Mittelung der Merkmale, eine Schicht mit 512 Neuronen und acht Klassenwahrscheinlichkeiten. Wir geben drei letzte geeignete Schichten zum Fine-Tuning frei. Wie im Original werden SGD und Datenerweiterung verwendet. Das Erstellen dieses Netzes ist bereitgestellter Code.

**Vor dem Start:** Welche Klassen könnten verwechselt werden? Der folgende Trainingsaufruf kann lange laufen. Beim nächsten Öffnen wird ein passender Lauf geladen; mit `TRAINING_STARTEN=True` wird ein unvollständiger Lauf fortgesetzt. Für ein **neues** Live-Training auch `LAUFGRUPPE` auf einen eigenen Namen setzen, z. B. `"demo_montag"`. Zum vorbereiteten Vergleich mit `LAUFGRUPPE="workshop_v1"` und `TRAINING_STARTEN=False` neu von oben starten, sobald kein eigener Lauf mehr läuft.

In [ ]:
modell1, verlauf1 = lauf("baseline", daten["x_train"], daten["y_train"])


### Baseline auswerten

**Konfusionsmatrix:** Zeile = wahres Label, Spalte = Vorhersage. **Klassenweise Kalibrierung:** Für jede Klasse betrachten wir deren vorhergesagte Wahrscheinlichkeit. Von Bildern mit etwa 0,6 für diese Klasse sollten ungefähr 60 Prozent tatsächlich zu ihr gehören. Das ist etwas anderes als nur die höchste Konfidenz jeder Vorhersage zu betrachten.

Accuracy zählt alle Treffer. Macro-F1 berechnet F1 über den gesamten Testsatz je Klasse und mittelt gleichgewichtet über alle acht Klassen.

In [ ]:
wahrscheinlichkeiten1 = werte_aus(projekt, LAUFGRUPPE + "_baseline", modell1,
                               verlauf1, daten["x_test"], daten["y_test"], klassennamen)


### Aufgabe 1: Viele Vorhersagen gleichzeitig

Jede Tabellenzeile gehört zu einem Bild. Die drei Spalten sind drei Übungsklassen. `axis=1` bedeutet: in jeder Zeile über die Klassen suchen. Bestimme je Bild die gewählte Klasse, ihre Konfidenz und ob sie zum Label passt.

**Vorwissen:** Grundlagen, Abschnitt 5 / Code-Challenge 3; Python-/ML-Kompass „Vorhersagen prüfen“.

**Ergebnis:** Ergänze die markierten Zeilen und lies die Rückmeldung. Bei offenen Antworten laufen die Modellvergleiche mit dem bereitgestellten Code weiter.

In [ ]:
beispiel = np.array([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]])
wahre_labels = np.array([1, 2])
klassen = None       # TODO: Position des größten Werts in jeder Zeile
konfidenzen = None   # TODO: Größter Wert in jeder Zeile
richtig = None      # TODO: Klassen mit wahren Labels vergleichen
kontrolliere("1a Klassen", klassen, np.array([1, 0]))
kontrolliere("1b Konfidenzen", konfidenzen, np.array([0.7, 0.8]))
kontrolliere("1c Korrektheit", richtig, np.array([True, False]))


<details>
<summary>Hinweis zur Idee</summary>

Du hast dieselben drei Operationen im Grundlagenkurs für ein Bild verwendet. Jetzt wendest du sie zeilenweise auf mehrere Bilder an.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

`np.argmax(tabelle, axis=1)` liefert die Positionen; `np.max(tabelle, axis=1)` die Werte. Mit `==` kannst du die beiden Label-Arrays direkt vergleichen.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
beispiel = np.array([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]])



wahre_labels = np.array([1, 2])
klassen = np.argmax(beispiel, axis=1)  # TODO: Position des größten Werts in jeder Zeile
konfidenzen = np.max(beispiel, axis=1) # TODO: Größter Wert in jeder Zeile
richtig = klassen == wahre_labels     # TODO: Klassen mit wahren Labels vergleichen
kontrolliere("1a Klassen", klassen, np.array([1, 0]))
kontrolliere("1b Konfidenzen", konfidenzen, np.array([0.7, 0.8]))
kontrolliere("1c Korrektheit", richtig, np.array([True, False]))
```

Für Bild 2 ist die Konfidenz 0,8, aber die Vorhersage falsch. Hohe Konfidenz garantiert keine Korrektheit.

</details>

## 4. Klassenungleichgewicht untersuchen

Wir behalten nur einen kleinen Anteil einer gewählten Klasse im Training. Alle anderen Trainingsbilder bleiben erhalten; Validierung und Test werden nicht verändert. Vermutet vorher: Welche Fehler könnten zunehmen? Die folgenden Einstellungen werden vor dem Lauf festgelegt.

In [ ]:
SELTENE_KLASSE = 0
BEHALTENER_ANTEIL = 0.1
x_schief, y_schief = hv.make_imbalanced(daten["x_train"], daten["y_train"],
                                      minority_class=SELTENE_KLASSE,
                                      keep_fraction=BEHALTENER_ANTEIL)
print("Vorher:", np.bincount(daten["y_train"], minlength=8))
print("Nachher:", np.bincount(y_schief, minlength=8))
modell2, verlauf2 = lauf("unausgewogen", x_schief, y_schief)
wahrscheinlichkeiten2 = werte_aus(projekt, LAUFGRUPPE + "_unausgewogen", modell2,
                               verlauf2, daten["x_test"], daten["y_test"], klassennamen)


### Aufgabe 2: Seltene Klassen stärker gewichten

Für Klasse k verwenden wir Gewicht = Gesamtzahl / (Klassenanzahl × Häufigkeit_k). Bei 150 Bildern und Häufigkeiten `[100, 50]` ergibt das `[0.75, 1.5]`. Die seltene Klasse zählt pro Beispiel doppelt so stark. Berechne jetzt die Gewichte für unsere acht Klassen. NumPy rechnet die Division für jeden Eintrag von `haeufigkeiten` einzeln.

**Vorwissen:** Oberstufen-Verhältnisrechnung; Einführung unmittelbar über dieser Aufgabe und Klassen-Gewichte-Folie.

**Ergebnis:** Ergänze die markierten Zeilen und lies die Rückmeldung. Bei offenen Antworten laufen die Modellvergleiche mit dem bereitgestellten Code weiter.

In [ ]:
haeufigkeiten = np.bincount(y_schief, minlength=8)
gesamt = len(y_schief)
gewichte = None  # TODO: Gesamtzahl / (8 * Häufigkeiten)
kontrolliere("2 Klassen-Gewichte", gewichte, gesamt / (8 * haeufigkeiten))


<details>
<summary>Hinweis zur Idee</summary>

Setze die bereitgestellten Größen in die Formel ein. Je kleiner die Häufigkeit im Nenner, desto größer wird das Gewicht.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

`gesamt / (8 * haeufigkeiten)` arbeitet mit dem ganzen Zahlenarray.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
haeufigkeiten = np.bincount(y_schief, minlength=8)
gesamt = len(y_schief)
gewichte = gesamt / (8 * haeufigkeiten)  # TODO: Gesamtzahl / (8 * Häufigkeiten)
kontrolliere("2 Klassen-Gewichte", gewichte, gesamt / (8 * haeufigkeiten))
```

Die Mathematik ist eine bekannte Verhältnisrechnung. `np.bincount` wird bereitgestellt; seine Implementierung musst du nicht kennen.

</details>

## 5. Klassen-Gewichte im Modellvergleich

Vor dem Lauf: Welche Änderung erwartest du für die seltene Klasse? Der bereitgestellte Code berechnet und übergibt die Gewichte tatsächlich an das Training. Vergleiche anschließend auch Fehler anderer Klassen. Eine Verbesserung ist nicht garantiert.

In [ ]:
klassengewichte = hv.class_weights(y_schief, num_classes=8)
print(klassengewichte)
modell3, verlauf3 = lauf("gewichtet", x_schief, y_schief, class_weight=klassengewichte)
wahrscheinlichkeiten3 = werte_aus(projekt, LAUFGRUPPE + "_gewichtet", modell3,
                               verlauf3, daten["x_test"], daten["y_test"], klassennamen)


### Aufgabe 3: Focal Loss aus bekannten Bausteinen

Wir verwenden wie in den Grundlagen genau eine Vorhersage: `y_true` ist der One-Hot-Vektor der wahren Klasse. Berechne `p_t`, `alpha_t` und `-alpha_t * (1-p_t)**gamma * log(p_t)`. Die spätere Verarbeitung vieler Bilder übernimmt der vorbereitete Trainingscode.

**Vorwissen:** Grundlagen, Abschnitt 7 / Code-Challenge 6; Focal-Loss-Folie; Python-/ML-Kompass „Loss verstehen“.

**Ergebnis:** Ergänze die markierten Zeilen und lies die Rückmeldung. Bei offenen Antworten laufen die Modellvergleiche mit dem bereitgestellten Code weiter.

In [ ]:
y_true = tf.constant([0., 1., 0.])
y_pred = tf.constant([0.1, 0.6, 0.3])
alpha = tf.constant([1., 2., 1.])
gamma = 2.
p_t = None        # TODO: Wahrscheinlichkeit der wahren Klasse auswählen und summieren
alpha_t = None    # TODO: Gewicht der wahren Klasse auswählen und summieren
focal_wert = None # TODO: Focal-Loss-Formel einsetzen
kontrolliere("3a p_t", p_t, .6)
kontrolliere("3b alpha_t", alpha_t, 2.)
kontrolliere("3c Focal Loss", focal_wert, -2 * (1-.6)**2 * np.log(.6))


<details>
<summary>Hinweis zur Idee</summary>

Die 1 im One-Hot-Vektor wählt den passenden Eintrag aus. Die anderen Einträge werden bei der Multiplikation null. Danach summierst du.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

`tf.reduce_sum(a * b)` verbindet Auswahl und Summe. `**` potenziert, `tf.math.log` berechnet den natürlichen Logarithmus.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
y_true = tf.constant([0., 1., 0.])
y_pred = tf.constant([0.1, 0.6, 0.3])
alpha = tf.constant([1., 2., 1.])
gamma = 2.
p_t = tf.reduce_sum(y_true * y_pred)  # TODO: Wahrscheinlichkeit der wahren Klasse auswählen und summieren
alpha_t = tf.reduce_sum(y_true * alpha) # TODO: Gewicht der wahren Klasse auswählen und summieren
focal_wert = -alpha_t * (1-p_t)**gamma * tf.math.log(p_t) # TODO: Focal-Loss-Formel einsetzen
kontrolliere("3a p_t", p_t, .6)
kontrolliere("3b alpha_t", alpha_t, 2.)
kontrolliere("3c Focal Loss", focal_wert, -2 * (1-.6)**2 * np.log(.6))
```

Hier ist die wahre Klasse an Position 1: p_t=0,6 und alpha_t=2. Der Faktor (1-p_t)^gamma dämpft den Loss bei großen p_t stärker.

</details>

## 6. Focal Loss im Modellvergleich

Wir verwenden die Focal-Loss-Definition der wahren Klasse aus Aufgabe 3. Alpha enthält die Klassen-Gewichte, Gamma ist 2. Bei Gamma 0 ergibt sich gewichtete Kreuzentropie. Das ist eine Gegenmaßnahme, kein Fairnessnachweis. Welche Fehler werden seltener, welche häufiger?

In [ ]:
alpha_modell = np.array([klassengewichte[i] for i in range(8)], dtype="float32")
modell4, verlauf4 = lauf("focal", x_schief, y_schief, loss="focal", alpha=alpha_modell, gamma=2.)
wahrscheinlichkeiten4 = werte_aus(projekt, LAUFGRUPPE + "_focal", modell4,
                               verlauf4, daten["x_test"], daten["y_test"], klassennamen)


## 7. Bildbeispiele und Grad-CAM vergleichen

Der Filtercode ist bereitgestellt. Er sucht (1) Bilder, die nur Modell 2 falsch einordnet und (2) Bilder, die alle Modelle richtig einordnen, bei denen sich ihre Konfidenzen aber unterscheiden. Wenn ein solcher Fall nicht vorkommt, wird das ausdrücklich gemeldet. Ein anderes Beispiel darf dann untersucht werden, erfüllt aber nicht den gesuchten Filter. Stellt unten `FALL` zunächst auf `"case1"`, danach auf `"case2"` und führt die Zelle erneut aus, um beide Fragestellungen zu vergleichen.

Grad-CAM hebt Regionen für eine gewählte Zielklasse hervor. Wir verwenden für alle vier Modelle dieselbe wahre Klasse als Ziel, damit die Heatmaps dieselbe Frage beantworten. Jede Karte wird separat skaliert: Vergleicht die Positionen der hervorgehobenen Bereiche, nicht ihre absolute Helligkeit. Heatmaps beweisen weder Ursache noch medizinische Relevanz.

In [ ]:
vorhersagen = [wahrscheinlichkeiten1, wahrscheinlichkeiten2, wahrscheinlichkeiten3, wahrscheinlichkeiten4]
auswahl = hv.choose_examples(vorhersagen, daten["y_test"])
for fall in ("case1", "case2"):
    print(fall, auswahl[fall][:5])
    if auswahl[fall + "_missing"]:
        print("Kein passendes Beispiel. Ersatz ist ausdrücklich kein Treffer dieses Filters.")
FALL = "case1"  # Danach "case2" wählen und diese Zelle erneut ausführen.
if FALL not in ("case1", "case2"):
    raise ValueError('FALL muss "case1" oder "case2" sein.')
beispiel_index = int(auswahl[FALL][0]) if len(auswahl[FALL]) else auswahl["fallback"]
print("Untersuchtes Bild:", beispiel_index)
for nummer, modell in enumerate((modell1, modell2, modell3, modell4), 1):
    print("Modell", nummer)
    hv.plot_example(modell, daten["x_test"][beispiel_index],
                    int(daten["y_test"][beispiel_index]), klassennamen,
                    target_class=int(daten["y_test"][beispiel_index]))


## 8. Unsere Beobachtung für den Science Slam

Haltet eine Beobachtung und eine offene Frage fest. Welche Idee könnte man einem Publikum mit einem kurzen Beispiel erklären? Der gemeinsame Science Slam dauert insgesamt zehn Minuten. Themen und Arbeitsauftrag stehen in [README – Science Slam](../README.md#science-slam).

<details>
<summary>Bereitgestellten technischen Code ansehen</summary>

Datenaufteilung, Metriken, Modellbau und Grad-CAM stehen in `src/hauptversuch.py`. Lokale Trainingsstände verwaltet `src/versuchsbetrieb.py`. Diese Implementierungen sind keine Schüleraufgaben.

</details>